# Check changes in units between Samuel-2 and Samuel-3

We have two extracts from SSNAP for Samuel-2 and Samuel-3, and we expect the stroke units to be different between them. Here we check which units seem to have closed, opened, or been renamed between the extracts.

Expected changes between Samuel-2 and Samuel-3 are as follows (Martin J's comments from August 2026):

| Area | Comment | Samuel-2 only | Samuel-2 & Samuel-3 | Samuel-3 only |
| --- | --- | --- | --- | --- |
| Bristol | New unit 'Bristol and Weston SU' formed from the merger of acute stroke services from three previous sites: North Bristol Hospital, Bristol Royal Infirmary (? known in Samuel-2 as University Hospitals Bristol Inpatient Team) and Weston super Mare (Weston General Hospital). | 'North Bristol Hospitals', 'Weston General Hospital' | 'University Hospitals Bristol Inpatient Team' | 'Bristol and Weston SU' |
| Cheltenham & Gloucester | acute stroke services moved from the Gloucester Royal Hospital to the Cheltenham Hospital | 'Gloucestershire Royal Hospital' | -- | 'Cheltenham & Gloucester Hospitals' | 
| Kent | formed from the merger of acute stroke services from Queen Elizabeth the Queen Mother Hospital, William Harvey Hospital and Invicta Ward Kent and Canterbury Hospital. | 'Queen Elizabeth the Queen Mother Hospital', 'William Harvey Hospital', 'Invicta Ward Kent and Canterbury Hospital' | -- | 'East Kent HASU' |
| Birmingham | This was formed by the merger of stroke services from Sandwell Hospital and Birmingham City Hospital, quite recently so there may be Sandwell and/or City patients in the cohort. Note: no Birmingham City Hospital in Samuel-2 or Samuel-3. | -- | 'Sandwell District Hospital' | 'Midland Metropolitan University Hospital (MMUH) - Acute Stroke Services' |
| Wales | Royal Glamorgan is the merger of Prince Charles and Prince Philip hospitals | 'Prince Charles Hospital' | 'Prince Philip Hospital' | 'Royal Glamorgan' |

## Code setup

In [1]:
# Import packages
import numpy as np
import os
import pandas as pd

from dataclasses import dataclass

## Set up paths and filenames

In [2]:
@dataclass(frozen=True)
class Paths:
    '''Singleton object for storing paths to data and database.'''

    team_code_sam2_path = '~/stroke-modelling/stroke-utilities/stroke_utilities/samuel2/data/'
    team_code_sam2_filename = 'team_code.csv'
    team_code_sam3_path = '~/samuel3/data/anon/'
    team_code_sam3_filename = 'stroke_team_id_map.csv'
    data_read_path: str = '~/samuel3/data/'
    data_read_filename: str = './cleaned_data_anon_teams(in).csv'
    data_read_path_sam2: str = '~/ssnap_data/'
    data_read_filename_sam2: str = './clean_samuel_ssnap_extract_v2.csv'
    data_save_path: str = 'data'
    team_present_save_filename: str = 'teams_present_samuel2_samuel3.csv'

paths = Paths()

## Load data

Samuel-2 team codes and hospital names:

In [4]:
p = os.path.join(paths.team_code_sam2_path, paths.team_code_sam2_filename)
df_old = pd.read_csv(p, index_col=0)

df_old = df_old.rename(columns={'team_code': 'code_samuel2'})

Samuel-3 team codes and hospital names:

In [10]:
p = os.path.join(paths.team_code_sam3_path, paths.team_code_sam3_filename)
df_new = pd.read_csv(p, index_col=0)

df_new = df_new.rename(columns={'stroke_team_id': 'code_samuel3'})

Gather codes and convert to simple yes/no are they present in each dataset:

In [12]:
df_present = pd.merge(
    df_old, df_new, left_index=True, right_index=True, how='outer')
df_present = df_present.notna().astype(int)
df_present = df_present.rename(
    columns={'code_samuel2': 'samuel2', 'code_samuel3': 'samuel3'})

# Save a copy:
p = os.path.join(paths.data_save_path, paths.team_present_save_filename)
df_present.to_csv(p)

In [13]:
df_present.head()

,samuel2,samuel3
stroke_team,,
Addenbrooke's Hospital,1,1
Basildon University Hospital,1,1
Blackpool Victoria Hospital,1,1
Bradford and Airedale SU,1,1
Bristol and Weston SU,0,1


## Check whether units are present in Samuel-2 or Samuel-3

In [14]:
hospitals_sam2 = df_present[df_present['samuel2'] == 1].index.values
hospitals_sam3 = df_present[df_present['samuel3'] == 1].index.values
# All hospitals:
hospitals_all = list(set(hospitals_sam2) | set(hospitals_sam3))
# Hospitals in both Samuel-2 and Samuel-3 data:
hospitals_both = list(set(hospitals_sam2) & set(hospitals_sam3))
# Hospitals in only one dataset:
hospitals_missing_in_samuel3 = list(set(hospitals_sam2) - set(hospitals_sam3))
hospitals_new_for_samuel3 = list(set(hospitals_sam3) - set(hospitals_sam2))

Visually inspect hospitals in one dataset and not the other:

In [15]:
hospitals_missing_in_samuel3

['Princess Of Wales Hospital',
 'Gloucestershire Royal Hospital',
 'Queen Elizabeth the Queen Mother Hospital',
 'North Bristol Hospitals',
 'Invicta Ward Kent and Canterbury Hospital',
 'William Harvey Hospital',
 'Prince Charles Hospital',
 'Weston General Hospital']

In [16]:
hospitals_new_for_samuel3

['Cheltenham & Gloucester Hospitals',
 'East Kent HASU',
 'Midland Metropolitan University Hospital (MMUH) - Acute Stroke Services',
 'Royal Glamorgan',
 'Bristol and Weston SU']

## Check for overlap in patients

Do teams that have merged have distinct timeframes? Is there some date after which only the new name is used?

Load in Samuel-3 data:

In [17]:
filename = os.path.join(paths.data_read_path, paths.data_read_filename)
data = pd.read_csv(filename)
data.shape

(452863, 70)

In [18]:
data.index.name = 'id'
data = data.reset_index()

Convert team codes to team names:

In [19]:
data['stroke_team'] = data['stroke_team'].map(dict(zip(df_new['code_samuel3'].values, df_new.index)))

Load in Samuel-2 data:

In [20]:
filename = os.path.join(paths.data_read_path_sam2, paths.data_read_filename_sam2)
data_sam2 = pd.read_csv(filename)
data_sam2.shape

(358993, 70)

Function to check the months and years used by selected teams in the data:

In [21]:
def find_timeframes_for_teams(teams_to_check, data_sam2, data):
    """
    Make dataframe of time range of Sam-2 and Sam-3 data for given teams.

    Inputs
    ------
    teams_to_check - list. Which teams to pull out the timeframes for.
    data_sam2      - pd.DataFrame. Full SSNAP data for Samuel-2.
    data           - pd.DataFrame. Full SSNAP data for Samuel-3.

    Returns
    -------
    df_times - pd.DataFrame. Row for each stroke unit, columns for the
               dates of first and last admissions in YYYY/MM format.
    """
    # Set up results df.
    # Explicitly set columns to prevent FutureWarning.
    # Later use loc to make new column to contain string but cannot
    # set dtype as loc is happening, so get warning.
    cols = ['samuel-2_first', 'samuel-2_last',
            'samuel-3_first', 'samuel-3_last']
    df_times = pd.DataFrame(columns=cols, dtype=object)

    # Fill results df for each team in turn:
    for team in teams_to_check:
        for sam_key, d in {'samuel-2': data_sam2, 'samuel-3': data}.items():
            # Look up data for this team:
            mask = d['stroke_team'] == team
            data_here = (
                d.loc[mask, ['year', 'month']].value_counts().sort_index())
            # Format data for storing in results df:
            if len(data_here) > 0:
                first_here = data_here.index[0]
                last_here = data_here.index[-1]
                # Condense year and month into one column:
                first_here = f'{first_here[0]}/{first_here[1]:02}'
                last_here = f'{last_here[0]}/{last_here[1]:02}'
            else:
                # Easier to read this character than <NA>
                first_here = '-'
                last_here = '-'
            # Store results:
            df_times.loc[team, [f'{sam_key}_first']] = first_here
            df_times.loc[team, [f'{sam_key}_last']] = last_here
    return df_times

Run the function for all teams:

In [22]:
df_times = find_timeframes_for_teams(hospitals_all, data_sam2, data)

## Units suspected to have merged

Display timeframes for questionable geographic areas:

In [23]:
team_sets = dict(
    birm = [
        'Sandwell District Hospital',
        'Midland Metropolitan University Hospital (MMUH) - Acute Stroke Services',
        ],
    glouc = [
        'Gloucestershire Royal Hospital',
        'Cheltenham & Gloucester Hospitals',
        ],
    wales = [
        'Princess Of Wales Hospital',
        'Prince Philip Hospital',
        'Prince Charles Hospital',
        'Royal Glamorgan'
        ],
    kent = [
        'Invicta Ward Kent and Canterbury Hospital',
        'Queen Elizabeth the Queen Mother Hospital',
        'William Harvey Hospital',
        'East Kent HASU',
        ],
    bristol = [
        'North Bristol Hospitals',
        'Weston General Hospital',
        'University Hospitals Bristol Inpatient Team',
        'Bristol and Weston SU',
        ],
)

In [24]:
for label, teams in team_sets.items():
    print(label)
    display(df_times.loc[teams])

birm


,samuel-2_first,samuel-2_last,samuel-3_first,samuel-3_last
Sandwell District Hospital,2016/01,2021/12,2020/01,2024/09
Midland Metropolitan University Hospital (MMUH) - Acute Stroke Services,-,-,2024/10,2025/12


glouc


,samuel-2_first,samuel-2_last,samuel-3_first,samuel-3_last
Gloucestershire Royal Hospital,2016/01,2021/12,-,-
Cheltenham & Gloucester Hospitals,-,-,2020/01,2025/12


wales


,samuel-2_first,samuel-2_last,samuel-3_first,samuel-3_last
Princess Of Wales Hospital,2016/01,2021/12,-,-
Prince Philip Hospital,2016/01,2021/12,2020/01,2025/12
Prince Charles Hospital,2016/01,2021/12,-,-
Royal Glamorgan,-,-,2020/01,2025/12


kent


,samuel-2_first,samuel-2_last,samuel-3_first,samuel-3_last
Invicta Ward Kent and Canterbury Hospital,2020/03,2021/12,-,-
Queen Elizabeth the Queen Mother Hospital,2016/01,2021/07,-,-
William Harvey Hospital,2016/01,2021/05,-,-
East Kent HASU,-,-,2020/04,2025/12


bristol


,samuel-2_first,samuel-2_last,samuel-3_first,samuel-3_last
North Bristol Hospitals,2016/01,2021/12,-,-
Weston General Hospital,2016/01,2021/12,-,-
University Hospitals Bristol Inpatient Team,2016/01,2021/12,2020/01,2023/03
Bristol and Weston SU,-,-,2020/01,2025/12


Results:
+ Sandwell's final entries are for September 2024. Midland Met's first entries are for October 2024. Looks like the same team, but they didn't retroactively rename the Sandwell data.
+ University Bristol's final entries are for March 2023. Bristol and Weston's first entries are for January 2020. Assume unrelated.
+ Prince Philip and Royal Glamorgan both span the full range of the Samuel-3 data (January 2020 to December 2025 inclusive). Assume unrelated.
+ East Kent spans nearly the full range (April 2020 to December 2025). But it's not in Samuel-2, so presumably those other units were renamed for the Samuel-3 extract. April is suspicious - first three months of data gone missing?
+ Cheltenham and Gloucester spans the full range (January 2020 to December 2025). But again it's not in Samuel-2, so the Gloucester data must have been renamed. Since there's only one Samuel-2 unit, can use the same code unambiguously.

## Other odd units

The full data range is 2016/01-2021/12 for Samuel-2 and 2020/01-2025/12 for Samuel-3.

Pick out teams that span the full date ranges:

In [25]:
s2f, s2l, s3f, s3l = '2016/01', '2021/12', '2020/01', '2025/12'

In [26]:
m = ((df_times['samuel-2_first'] == s2f) &
     (df_times['samuel-2_last'] == s2l))
df_times['samuel-2_full_span'] = 0
df_times.loc[m, 'samuel-2_full_span'] = 1

m = ((df_times['samuel-3_first'] == s3f) &
     (df_times['samuel-3_last'] == s3l))
df_times['samuel-3_full_span'] = 0
df_times.loc[m, 'samuel-3_full_span'] = 1

In [27]:
m = ((df_times['samuel-3_full_span'] == 1) &
     (df_times['samuel-3_full_span'] == 1))
hospitals_full_span_23 = df_times[m].index
hospitals_not_full_span_23 = [
    h for h in df_times.index if h not in hospitals_full_span_23]
m = ((df_times['samuel-3_full_span'] == 1) &
     (df_times['samuel-3_full_span'] == 0))
hospitals_full_span_3 = df_times[m].index
m = ((df_times['samuel-3_full_span'] == 0) &
     (df_times['samuel-3_full_span'] == 1))
hospitals_full_span_2 = df_times[m].index

len(hospitals_full_span_23), len(hospitals_full_span_3), len(hospitals_full_span_2)

(106, 0, 0)

Check hospitals that don't span the full date ranges and aren't in the above questionable geographic areas:

In [28]:
# List of all teams in the questionable geographic areas:
h_odd = sum(team_sets.values(), [])
# List of teams with incomplete date range and not in above list:
h = [h for h in hospitals_not_full_span_23 if h not in h_odd]

df_times.loc[h]

,samuel-2_first,samuel-2_last,samuel-3_first,samuel-3_last,samuel-2_full_span,samuel-3_full_span
Medway Maritime Hospital,2016/01,2020/04,2020/01,2020/04,0,0
Royal Liverpool University Hospital,2016/01,2021/12,2020/01,2022/10,1,0
Grange University Hospital,2020/09,2021/12,2020/11,2025/12,0,0
University Hospitals Dorset Stroke Service,2020/10,2021/12,2020/09,2025/12,0,0
Queen's Medical Centre - Nottingham,2017/09,2021/12,2020/07,2025/12,0,0
Warwick Hospital,2016/01,2021/12,2020/01,2022/08,1,0


Straightforward cases:
+ Royal Liverpool University Hospital - presumably closed in October 2022.
+ University Hospitals Dorset Stroke Service - presumably didn't open until September or October 2020.
+ Medway Maritime Hospital - presumably closed in April 2020.
+ Warwick Hospital - presumably closed in August 2022.

Oddities:
+ Queen's Medical Centre - Nottingham. Where is the January-June 2020 data? Presumably present in Samuel-2, missing in Samuel-3. Check this.
+ Grange University Hospital - where is the September and October 2020 data in Samuel-3?

Check the oddities more closely.

### Nottingham

Check whether Nottingham has data for all months:

In [29]:
m = ((data_sam2['stroke_team'] == "Queen's Medical Centre - Nottingham") &
     (data_sam2['year'] < 2021))
data_sam2.loc[m, ['year', 'month']].value_counts().sort_index()

year  month
2017  9          1
      10         1
2018  2          2
      3          2
      8          1
      9          1
      10         1
      12         3
2019  3          1
      4          2
      8          1
2020  1          1
      2          3
      5          2
      6          3
      7         53
      8         63
      9         77
      10        83
      11        94
      12       100
dtype: int64

Very low numbers of patients in Nottingham until July 2020. Presumably there's some operational reason such as the unit being temporarily closed until July 2020.

Not concerned about this.

### Grange University Hospital

Check what's going on in September and October 2020:

In [30]:
m = ((data_sam2['stroke_team'] == 'Grange University Hospital') &
    (data_sam2['year'] < 2022))
data_sam2.loc[m, ['year', 'month']].value_counts().sort_index()

year  month
2020  9         2
      10        2
      11       22
      12       51
2021  1        35
      2        38
      3        45
      4        25
      5        33
      6        28
      7        39
      8        34
      9        37
      10       34
      11       36
      12       32
dtype: int64

Again it looks like this unit didn't open in earnest until November 2020, which is when its data begins in the Samuel-3 extract.

Not concerned about this.

## Conclusion

The data suggests that the stroke units have merged as expected:

| Area | Samuel-2 only | Samuel-2 & Samuel-3 | Samuel-3 only |
| --- | --- | --- | --- |
| Bristol | 'North Bristol Hospitals', 'Weston General Hospital' | 'University Hospitals Bristol Inpatient Team' | 'Bristol and Weston SU' |
| Cheltenham & Gloucester | 'Gloucestershire Royal Hospital' | -- | 'Cheltenham & Gloucester Hospitals' |
| Kent | 'Queen Elizabeth the Queen Mother Hospital', 'William Harvey Hospital', 'Invicta Ward Kent and Canterbury Hospital' | -- | 'East Kent HASU' |
| Birmingham | -- | 'Sandwell District Hospital' | 'Midland Metropolitan University Hospital (MMUH) - Acute Stroke Services' |
| Wales | 'Prince Charles Hospital' | 'Prince Philip Hospital' | 'Royal Glamorgan' |

The data suggests that some units opened or closed during the Samuel-3 extract timespan.

Closed:
+ Royal Liverpool University Hospital - presumably closed in October 2022.
+ Medway Maritime Hospital - presumably closed in April 2020.
+ Warwick Hospital - presumably closed in August 2022.

Opened:
+ University Hospitals Dorset Stroke Service - presumably opened in September 2020.
+ Queen's Medical Centre - Nottingham - presumably opened in July 2020.
+ Grange University Hospital - presumably opened in November 2020.
